<!-- 
 © Copyright IBM Corporation 2025
 SPDX-License-Identifier: Apache-2.0
 -->

# TerraKit: Climate Data Store search and query

Use the TerraKit Climate Data Store Data Connector to search and query data from [CDS](https://cds.climate.copernicus.eu/datasets/). Available collections include:

 - [ERA5 post-processed daily statistics on single levels from 1940 to present](https://cds.climate.copernicus.eu/datasets/derived-era5-single-levels-daily-statistics?tab=overview)
 - [CORDEX regional climate model data on single levels](https://cds.climate.copernicus.eu/datasets/projections-cordex-domains-single-levels?tab=overview)

<div class="alert alert-block alert-success">
<b>Install TerraKit</b> For instructions on how to install TerraKit, take a look at the <a href="https://terrastackai.github.io/terrakit/">Welcome</a> page.
</div>


In [1]:
import numpy as np


from terrakit import DataConnector
from terrakit.download.transformations.impute_nans_xarray import impute_nans_xarray
from terrakit.download.transformations.scale_data_xarray import scale_data_xarray
from terrakit.download.geodata_utils import save_data_array_to_file

<!--
 Copyright (c) 2026 Copyright 2024 IBM Corp
 
 This software is released under the MIT License.
 https://opensource.org/licenses/MIT
-->

### Create an account
Create an account at [https://cds.climate.copernicus.eu/](https://cds.climate.copernicus.eu/). Once created, find your API  key under the `Profile` section. Each dataset may also require accepting the licence agreement. If this is the case, the first time a request is made, an error will be returned with the url to visit to accept the terms.


### Connect to CDS

In [2]:
data_connector = "climate_data_store"
dc = DataConnector(connector_type=data_connector)
dc.connector.list_collections()

2026-03-16 19:12:13,321 - INFO - Initializing DataConnector with connector type: climate_data_store
2026-03-16 19:12:13,321 - INFO - climate_data_store
2026-03-16 19:12:13,323 - INFO - climate_data_store
2026-03-16 19:12:13,325 - INFO - Listing available collections


['projections-cordex-domains-single-levels',
 'derived-era5-single-levels-daily-statistics']

<!--
 Copyright (c) 2026 Copyright 2024 IBM Corp
 
 This software is released under the MIT License.
 https://opensource.org/licenses/MIT
-->

TerraKit defines a bounding box as a list using the format `[min_lon, min_lat, max_lon, max_lat]`, which is equivalent to `[West, South, East, North]`. Let's do that!

In [3]:
bbox = [
    34.5,
    -0.5,
    35.0,
    0.0,
]  # [West, South, East, North] / [min_lon, min_lat, max_lon, max_lat]

### Derived ERA5 Single Levels Daily Statistics


Now we can search for data from the Derived ERA5 Single Levels Daily Statistics collection.

The `find_data()` function will search for data and return both a list of unique dates where data is available, but also the raw results from the search.

In [4]:
collection_name = "derived-era5-single-levels-daily-statistics"

bands = ["2m_temperature", "mean_total_precipitation_rate"]

unique_dates, results = dc.connector.find_data(
    data_collection_name=collection_name,
    date_start="2025-01-01",
    date_end="2025-01-02",
    bands=bands,
    bbox=bbox,
)

print(unique_dates)
print(results)

1940-01-01 00:00:00 2026-02-04 00:00:00
['2025-01-01', '2025-01-02']
[{'collection': 'derived-era5-single-levels-daily-statistics', 'date_range': '2025-01-01 to 2025-01-02', 'total_dates': 2, 'temporal_extent': {'interval': [['1940-01-01T00:00:00+00:00', '2026-02-04T00:00:00+00:00']]}, 'spatial_extent': {'bbox': [[0.0, -89.0, 360.0, 89.0]]}}]


Now to query the data, we specify the bands we want to return, plus an (optional) save filename.  The `get_data()` function will query the data from the data source and return an xarray object containing all the fetched data with dimensions (time, band, y, x). All dates are stacked along the time dimension, and all bands are stacked along the band dimension.

Optionally, if `save_file=` is provided as an argument to `get_data()`, individual GeoTIFF files will be saved for each date with the naming pattern: {save_file}_{date}.tif (e.g., 'output_2025-01-01.tif', 'output_2025-01-02.tif'). Each file contains all requested bands for that specific date.

In [6]:
save_filestem = f"./tmp_download/{data_connector}_{collection_name}"

da = dc.connector.get_data(
    data_collection_name=collection_name,
    date_start="2025-01-01",
    date_end="2025-01-02",
    bbox=bbox,
    bands=bands,
    save_file=f"{save_filestem}.tif",
)
dai = scale_data_xarray(da, list(np.ones(len(bands))))
dai = impute_nans_xarray(dai)
save_data_array_to_file(dai, save_file=f"{save_filestem}.tif", imputed=True)

2026-03-16 19:12:44,636 - INFO - Submitting CDS request for derived-era5-single-levels-daily-statistics
2026-03-16 19:12:44,637 - INFO - Date range: 2025-01-01 to 2025-01-02 (2 days)
2026-03-16 19:12:44,637 - INFO - Area: 3080.22 km²
2026-03-16 19:12:44,637 - INFO - Variables: 2
2026-03-16 19:12:44,637 - INFO - Estimated size: ~0.12 MB
2026-03-16 19:12:44,638 - INFO - Estimated time: ~2.0 minutes
2026-03-16 19:12:45,100 - INFO - Request submitted to CDS queue. Please wait...
2026-03-16 19:12:45,793 INFO Request ID is bbf3a88d-e41e-41ac-8213-b2ce3324b64f
2026-03-16 19:12:45,793 - INFO - Request ID is bbf3a88d-e41e-41ac-8213-b2ce3324b64f
2026-03-16 19:12:45,856 INFO status has been updated to accepted
2026-03-16 19:12:45,856 - INFO - status has been updated to accepted
2026-03-16 19:13:00,005 INFO status has been updated to successful
2026-03-16 19:13:00,005 - INFO - status has been updated to successful
2026-03-16 19:13:00,241 - INFO - Downloading https://object-store.os-api.cci2.ecmwf.

f8c113b5e91ec99cae15946945e28136.zip:   0%|          | 0.00/49.6k [00:00<?, ?B/s]

2026-03-16 19:13:01,332 - INFO - ✓ Download complete: cds_derived-era5-single-levels-daily-statistics_20260316_191245.zip
2026-03-16 19:13:01,332 - INFO - Actual time: 0.3 minutes
2026-03-16 19:13:01,345 - WARNING - Could not match NetCDF variable 't2m' to any requested band. Using NetCDF variable name 't2m' as band label. Requested bands: ['2m_temperature', 'mean_total_precipitation_rate']
2026-03-16 19:13:01,356 - INFO - Mapped NetCDF variable 'avg_tprate' to requested band 'mean_total_precipitation_rate' using attribute matching (long_name/standard_name)
2026-03-16 19:13:01,357 - INFO - Final band name mapping: {'t2m': 't2m', 'avg_tprate': 'mean_total_precipitation_rate'}
2026-03-16 19:13:01,398 - INFO - Saving cloud optimized geotiff to ./tmp_download/climate_data_store_derived-era5-single-levels-daily-statistics_2025-01-01.tif
2026-03-16 19:13:01,410 - INFO - Set band descriptions: ['mean_total_precipitation_rate', 't2m']
2026-03-16 19:13:01,411 - INFO - Saving cloud optimized geo